In [1]:
import os
import sys
import django


project_root = os.path.abspath("../..") 
if project_root not in sys.path:
    sys.path.append(project_root)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "scholaria.settings")  # Adjust 'scholaria' to your project settings folder name
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

from openai import OpenAI
from dotenv import load_dotenv
import os
from langchain_huggingface import HuggingFaceEmbeddings
from courses.models import Lesson


load_dotenv()
client = OpenAI(
    api_key=os.environ.get('GROQ_API_KEY'),
    base_url='https://api.groq.com/openai/v1'
)

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

INFO 2026-07-31 21:19:50,872 model 7896 126545863499904 No device provided, using cpu
INFO 2026-07-31 21:19:51,149 _client 7896 126545863499904 HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO 2026-07-31 21:19:51,398 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
WARNING 2026-07-31 21:19:51,399 _http 7896 126545863499904 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO 2026-07-31 21:19:51,487 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO 2026-07-31 21:19:51,711 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolv

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO 2026-07-31 21:19:53,423 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO 2026-07-31 21:19:53,611 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO 2026-07-31 21:19:53,846 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO 2026-07-31 21:19:54,052 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO 2026-07-31 21:19:54,258 _client 7896 126545863499904 HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 T

In [2]:
print('starting')

lessons_list = Lesson.objects.all()
print(lessons_list)
lessons_content = [f"\nTitle:{lesson.title} \nContent: {lesson.content}" for lesson in lessons_list]
evolution_lesson = Lesson.objects.get(title="The evolution of mathematics")
print(evolution_lesson.content)

print('embedding lessons data...')
vectors = embeddings.embed_documents(lessons_content)

for lesson, vector in zip(lessons_list, vectors):
    print(f"Lesson: {lesson.title} Vector: \n {vector}")
    lesson.embedding = vector

print('saving changes to database...')
Lesson.objects.bulk_update(
    lessons_list,
    fields=['embedding'],
    batch_size=100
)

print('Done!')




starting
<QuerySet [<Lesson: SQL antipatterns>, <Lesson: Testing for documents uploading>, <Lesson: Word Problems>, <Lesson: What is Multiplication?>, <Lesson: Lets see again>, <Lesson: SQL Antipatterns>, <Lesson: TESTING AGAIN!>, <Lesson: Measuring Length>, <Lesson: Let's see>, <Lesson: A quick history!>, <Lesson: Geography>, <Lesson: TEST>, <Lesson: The Times Tables>, <Lesson: The evolution of mathematics>, <Lesson: Introduction to Addition>, <Lesson: Fun with Subtraction>, <Lesson: Shapes and Angles>, <Lesson: Bushido Book ?>]>
<div _ngcontent-ng-c1751569727="" inline-copy-host="" class="markdown markdown-main-panel stronger enable-updated-hr-color" id="model-response-message-contentr_c0e2693e94fe1488" aria-live="polite" aria-busy="false" dir="ltr" style="--animation-duration: 400ms; --fade-animation-function: ease-out; animation: auto ease 0s 1 normal none running none; appearance: none; background: none 0% 0% / auto repeat scroll padding-box border-box rgba(0, 0, 0, 0); border-sty

In [3]:
def embed_input(input):
    vectorized_input = embeddings.embed_query(input)
    return vectorized_input


In [4]:
from pgvector.django import CosineDistance

def search(query):
    results = Lesson.objects.order_by(
        CosineDistance('embedding', query)
    )[:2]
    return results

search(embed_input("mathematics evolution"))

<QuerySet [<Lesson: Geography>, <Lesson: The Times Tables>]>

In [13]:
from utils.data_prepping import clean_text

def build_context(query: str) -> str:
    search_results = search(embed_input(query))
    results_content = [lesson.content for lesson in search_results]

    raw_text = "\n".join(results_content)

    processed_text = clean_text(raw_text)

    print(processed_text)

    return processed_text

def build_prompt(user_input : str):

    context = build_context(user_input)

    prompt = f"""
    
    You are an expert tutor. 
    
    Primary source your answer from the provided lesson context below. If the context does not fully cover the user's question, rely on your general knowledge to provide a complete answer, but explicitly note which parts come from outside the lesson.
    
    User question : {user_input}
    {context}
    """

    return prompt


In [14]:
def llm(model_name, query):
    prompt = build_prompt(query)
    res = client.responses.create(
        input=prompt,
        model=model_name
    )
    return res.output_text

In [15]:
llm("llama-3.1-8b-instant", "mathematics evolution")

While we often think of math as a series of discoveries, it is also a story of geography. Mathematics was never a linear progression in one place; it was a global relay race where ideas were passed, translated, and refined across borders and cultures.The Great Translation MovementsMathematics survived and thrived because of massive intellectual migrations.The Silk Road Exchange: Mathematical concepts, such as the abacus and early algebraic methods, traveled between China, India, and Persia.The House of Wisdom: In 9th-century Baghdad, scholars translated Greek, Indian, and Persian texts into Arabic. Without this "preservation project," much of the work by Euclid and Archimedes might have been lost to history during the European Dark Ages.The Toledo Connection: In 12th-century Spain, Christian, Jewish, and Muslim scholars worked together to translate these Arabic manuscripts into Latin, finally "re-introducing" advanced mathematics to Europe and sparking the Renaissance.
Let's memorize o

INFO 2026-07-31 21:27:28,710 _client 7896 126545863499904 HTTP Request: POST https://api.groq.com/openai/v1/responses "HTTP/1.1 200 OK"


'It seems like there are two separate topics being discussed here: the evolution of mathematics and the memorization of single-digit times tables.\n\nFirst, let\'s discuss the evolution of mathematics. Based on the provided context, we can see that mathematics has a rich history of global exchange and refinement. \n\nThis context highlights three significant "Great Translation Movements" that contributed to the advancement of mathematics:\n\n1. The Silk Road Exchange: Mathematical concepts were shared between China, India, and Persia.\n2. The House of Wisdom: Scholars in 9th-century Baghdad translated Greek, Indian, and Persian texts into Arabic, preserving ancient knowledge during the European Dark Ages.\n3. The Toledo Connection: Scholars in 12th-century Spain translated Arabic manuscripts into Latin, re-introducing advanced mathematics to Europe and sparking the Renaissance.\n\nTo summarize, the evolution of mathematics involved the exchange of ideas across cultures, borders, and ce